[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S09/2_Practica/para_casa/S09_03_recomendadores.ipynb)

# S09 · 03 — Recomendar experiencias a los huéspedes

Sprint 9 · M3 (UD 3.11, sistemas de recomendación)

**Contexto TurisData.** Además del alojamiento, TurisData vende **25 experiencias** (excursiones, talleres, deportes). Tenemos 2 232 valoraciones de 300 usuarios (de 1 a 5 estrellas). Cada usuario solo ha valorado entre 4 y 11 experiencias: la mayoría de las casillas de la tabla están **vacías**. Queremos decirle a cada cliente *"esto te puede gustar"*.

**Este cuaderno tiene 5 ejercicios y un mini-reto.** Los ejercicios 1, 2 y 3 son los mismos que los del cuaderno «en clase» (allí, del 1 al 3).

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

valoraciones = pd.read_csv(dato("valoraciones_experiencias.csv"))
experiencias = pd.read_csv(dato("experiencias.csv"))
print(valoraciones.shape, experiencias.shape)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Las primeras valoraciones

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas valoraciones hay y qué columnas tiene la tabla?
2. ¿Qué nota dio el usuario 1 a la experiencia 15?
3. ¿Por qué no hay una nota para cada pareja usuario-experiencia?

In [ ]:
valoraciones.head()

### 👀 A3 · EJEMPLO — Separar la prueba y construir la matriz

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Qué tamaño tiene la matriz `R` y qué porcentaje de casillas tiene dato?
2. ¿Qué nota media global sale?
3. ¿Por qué se esconde un 20 % de las valoraciones antes de construir la matriz?

In [ ]:
prueba = valoraciones.sample(frac=0.2, random_state=SEMILLA)
entrenamiento = valoraciones.drop(prueba.index)

# filas = usuarios 1..300, columnas = experiencias 1..25
R = entrenamiento.pivot(index="id_usuario", columns="id_experiencia", values="valoracion")
R = R.reindex(index=range(1, 301), columns=range(1, 26))
M = R.notna().values                       # True donde hay valoración (matriz de "visto")
print("Matriz:", R.shape, "· casillas con dato:", f"{M.mean():.0%}")

media_global = entrenamiento["valoracion"].mean()
print("Nota media global:", round(media_global, 2))

# posiciones (fila, columna) de las valoraciones de prueba, para evaluar
fila_p = prueba["id_usuario"].values - 1
col_p = prueba["id_experiencia"].values - 1
real_p = prueba["valoracion"].values

def evaluar_rmse(P):
    """RMSE de una matriz de predicciones P (300 x 25) sobre las valoraciones de prueba."""
    pred = np.clip(P[fila_p, col_p], 1, 5)
    return float(np.sqrt(np.mean((pred - real_p) ** 2)))

### 👀 A4 · EJEMPLO — El punto de partida: predecir siempre la media

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué RMSE tiene predecir siempre la media global?
2. ¿Qué significa ese número en palabras?
3. ¿Para qué sirve conocer este valor?

In [ ]:
P_global = np.full((300, 25), media_global)
print("RMSE prediciendo siempre la media global:", round(evaluar_rmse(P_global), 3))

### ✏️ A5 · EJERCICIO 1 de 5 — Popularidad: la media de cada experiencia

**Tu turno:** Calcula en `media_item` la nota media de cada experiencia **solo con la matriz de entrenamiento `R`** (una `Series` de 25 valores). Después crea `P_pop`, una matriz de 300 × 25 en la que **todas las filas** son `media_item` (`np.tile`), y guarda su error en `rmse_pop` con `evaluar_rmse`.

**Debe salir:** `RMSE popularidad = 1.029` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Qué RMSE tiene el recomendador por popularidad?
2. ¿Por qué mejora al de la media global?
3. ¿Por qué hay que calcular `media_item` con `R` y no con todas las valoraciones?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: la media por columnas de un DataFrame promedia cada experiencia; np.tile sirve para repetir un vector en varias filas
media_item = ...
P_pop = ...
rmse_pop = ...

assert len(media_item) == 25 and P_pop.shape == (300, 25), "media_item debe tener 25 valores y P_pop tamaño 300 x 25"
assert 0.95 < rmse_pop < 1.1, "El RMSE de popularidad debería rondar 1,0; revisa que uses R (solo entrenamiento)"
print(f"RMSE popularidad = {rmse_pop:.3f}")
print("✔ Ejercicio 1 correcto")

### 👀 A6 · EJEMPLO — Las cinco experiencias mejor valoradas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Cuál es la experiencia mejor valorada y qué nota media tiene?
2. ¿Qué diferencia hay entre la primera y la quinta de la lista?
3. ¿Qué le recomendaría este método a un usuario al que le encanta el mar?

In [ ]:
top = media_item.sort_values(ascending=False).head(5)
print(experiencias.set_index("id_experiencia").loc[top.index, "nombre"].to_frame().assign(nota=top.round(2)))

### ✏️ A7 · EJERCICIO 2 de 5 — La función precision@k

**Tu turno:** Completa la función `precision_k(P, k=3)`. Para cada usuario de `relevantes`: (1) toma su fila de `P`, (2) pon `-inf` en las experiencias que ya valoró en entrenamiento (fila de `M`), (3) quédate con las `k` de mayor puntuación (`np.argsort`; suma 1 para pasar de posición a `id_experiencia`) y (4) calcula qué fracción de esas `k` están en sus experiencias relevantes. Devuelve la media de todos los usuarios.

**Debe salir:** `Usuarios evaluables: 115`, `precision@3 popularidad = 0.125 · azar = 0.081` y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Cuántos usuarios son evaluables y por qué no son 300?
2. ¿Qué precision@3 tiene la popularidad y cuál el azar?
3. ¿Por qué se pone `-inf` en las experiencias que el usuario ya valoró?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
relevantes = prueba[prueba["valoracion"] >= 4]      # lo que "le gusta" a cada usuario
print("Usuarios evaluables:", relevantes["id_usuario"].nunique())

def precision_k(P, k=3):
    aciertos = []
    # TODO: Recorre relevantes.groupby("id_usuario"); copia la fila P[u-1]; usa M[u-1] para ocultar lo ya visto; compara con set(...)
    ...
    return float(np.mean(aciertos))

p_pop = precision_k(P_pop)
azar = precision_k(np.random.default_rng(SEMILLA).random((300, 25)))
assert 0.09 < p_pop < 0.16, "precision@3 de popularidad debería rondar 0,12; revisa que ocultes lo ya valorado"
assert azar < p_pop + 0.02, "Recomendar al azar no debería superar claramente a popularidad; revisa la función"
print(f"precision@3 popularidad = {p_pop:.3f} · azar = {azar:.3f}")
print("✔ Ejercicio 2 correcto")

### 👀 A8 · EJEMPLO — Características de las experiencias y residuos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A8).

1. ¿Qué tamaño tiene `F` y qué significan sus filas y columnas?
2. Si un usuario da un 5 a una experiencia con media 4,0, ¿qué residuo tiene?
3. ¿Por qué se trabaja con residuos y no con las notas directamente?

In [ ]:
F = pd.get_dummies(experiencias.set_index("id_experiencia")["categoria"]).reindex(range(1, 26)).values.astype(float)
print("F (experiencia x categoría):", F.shape, "· categorías:", experiencias["categoria"].nunique())

media_item_v = media_item.fillna(media_global).values      # nota media de cada experiencia
C = np.where(M, R.values - media_item_v[None, :], 0.0)     # residuo (0 donde no hay valoración)

### ✏️ A9 · EJERCICIO 3 de 5 — El perfil de gustos de cada usuario

**Tu turno:** Calcula `perfil_usuario` (300 × 7): para cada usuario y categoría, la **suma de sus residuos en esa categoría dividida entre (número de valoraciones en esa categoría + 3)**. El "+3" es prudencia: con pocas valoraciones no nos fiamos. Pista: `C @ F` suma residuos por categoría y `M @ F` cuenta valoraciones (convierte `M` a `float`).

**Debe salir:** ✔ Ejercicio 3 correcto (`perfil_usuario` tiene 300 filas y 7 columnas, con valores pequeños)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A9).

1. ¿Qué significa un valor de +0,34 en la columna `mar`?
2. ¿Por qué se suma 3 al denominador?
3. ¿Qué perfil tiene un usuario sin valoraciones en entrenamiento?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A9, **después de intentarlo**.

In [ ]:
# pista: numerador y denominador son dos productos de matrices con F; el "+3" va solo en el denominador
perfil_usuario = ...

assert perfil_usuario.shape == (300, 7), "perfil_usuario debe ser 300 usuarios x 7 categorías"
assert np.abs(perfil_usuario).max() < 3, "Los valores deben ser residuos pequeños (entre -3 y 3); revisa la división"
assert np.allclose(perfil_usuario[M.sum(1) == 0], 0), "Sin valoraciones el perfil debe ser 0"
print("✔ Ejercicio 3 correcto")

### 👀 A10 · EJEMPLO — Predicción por contenido y su error

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A10).

1. ¿Qué RMSE y precision@3 da el contenido?
2. ¿Es mejor o peor que popularidad (1,029 y 0,125)?
3. ¿Qué debe cambiar para que el contenido mejore?

In [ ]:
# Predicción por contenido: nota media de la experiencia + afinidad del usuario con su categoría
P_cont = media_item_v[None, :] + perfil_usuario @ F.T
print("RMSE contenido:", round(evaluar_rmse(P_cont), 3), "· precision@3:", round(precision_k(P_cont), 3))

### ✏️ A11 · EJERCICIO 4 de 5 — Experiencias parecidas al Teide

**Tu turno:** Calcula `S`, la matriz 25 × 25 de similitud coseno entre experiencias a partir de las **columnas de `C`** (`cosine_similarity(C.T)` ya está importado). Pon la diagonal a 0 (una experiencia no se ayuda a sí misma) con `np.fill_diagonal`. Después guarda en `similares_teide` los **nombres** de las 3 experiencias más parecidas al *Ascenso al Teide* (id 1, columna 0).

**Debe salir:** `parecidas al Teide: ['Paseo a caballo', 'Spa y termas', 'Parque acuático']` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A11).

1. ¿Cuáles son las 3 experiencias más parecidas al Teide y qué similitud tiene la primera?
2. ¿Qué indica que las similitudes sean tan bajas (0,19 como máximo)?
3. ¿Por qué se pone la diagonal a 0?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A11, **después de intentarlo**.

In [ ]:
# pista: las experiencias son columnas de C; ordena la fila 0 de S de mayor a menor similitud y recuerda que id = posición + 1
S = ...
# (pon la diagonal de S a 0 con np.fill_diagonal)
similares_teide = ...

assert S.shape == (25, 25) and np.allclose(S, S.T), "S debe ser 25 x 25 y simétrica"
assert np.allclose(np.diag(S), 0), "La diagonal de S debe ser 0"
assert len(similares_teide) == 3 and all(isinstance(n, str) for n in similares_teide), "similares_teide: lista con 3 nombres"
print("parecidas al Teide:", similares_teide)
print("✔ Ejercicio 4 correcto")

### 👀 A12 · EJEMPLO — Predicción item-item y user-user

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A12).

1. ¿Cuál de los dos tiene menor RMSE y cuál mayor precision@3?
2. ¿Mejoran a popularidad (RMSE 1,029)?
3. ¿Qué compara la similitud de item-item y qué la de user-user?

In [ ]:
# Predicción item-item: nota media + media ponderada de los residuos del usuario en experiencias parecidas
P_item = media_item_v[None, :] + (C @ S.T) / (M.astype(float) @ np.abs(S).T + 1e-9)

# Predicción user-user: lo mismo mirando usuarios parecidos
Su = cosine_similarity(C)
np.fill_diagonal(Su, 0)
P_user = media_item_v[None, :] + (Su @ C) / (np.abs(Su) @ M.astype(float) + 1e-9)

for nombre, P in (("item-item", P_item), ("user-user", P_user)):
    print(f"{nombre}: RMSE = {evaluar_rmse(P):.3f} · precision@3 = {precision_k(P):.3f}")

### 👀 A13 · EJEMPLO — La función factorizar y el efecto de k

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A13).

1. ¿Con qué k sale el menor RMSE y cuánto vale?
2. ¿Qué pasa con el RMSE cuando k pasa de 3 a 6?
3. ¿Qué son los «gustos ocultos» y por qué se llaman así?

In [ ]:
def factorizar(R, k=3, lam=5.0, iteraciones=30, semilla=SEMILLA):
    """Factorización con sesgos por mínimos cuadrados alternos. Devuelve la matriz completa de predicciones."""
    rng = np.random.default_rng(semilla)
    Mv = R.notna().values
    Y = np.nan_to_num(R.values)
    n_u, n_i = Y.shape
    mu = np.nanmean(R.values)
    P = rng.normal(0, 0.1, (n_u, k))          # gustos de cada usuario
    Q = rng.normal(0, 0.1, (n_i, k))          # intensidades de cada experiencia
    b_u, b_i = np.zeros(n_u), np.zeros(n_i)
    for _ in range(iteraciones):
        res = (Y - mu - b_i[None, :] - P @ Q.T) * Mv
        b_u = res.sum(1) / (Mv.sum(1) + 5)
        res = (Y - mu - b_u[:, None] - P @ Q.T) * Mv
        b_i = res.sum(0) / (Mv.sum(0) + 5)
        T = Y - mu - b_u[:, None] - b_i[None, :]
        for u in range(n_u):
            Qu = Q[Mv[u]]
            P[u] = np.linalg.solve(Qu.T @ Qu + lam * np.eye(k), Qu.T @ T[u, Mv[u]])
        for i in range(n_i):
            Pi = P[Mv[:, i]]
            Q[i] = np.linalg.solve(Pi.T @ Pi + lam * np.eye(k), Pi.T @ T[Mv[:, i], i])
    return mu + b_u[:, None] + b_i[None, :] + P @ Q.T

for k in (2, 3, 4, 6):
    print(f"k = {k} gustos ocultos → RMSE = {evaluar_rmse(factorizar(R, k=k)):.3f}")

### ✏️ A14 · EJERCICIO 5 de 5 — La factorización y la tabla comparativa

**Tu turno:** Guarda en `P_mf` la matriz de `factorizar(R, k=3)` y calcula `rmse_mf` y `p_mf` (precision@3). Después, en `mejor_rmse`, el nombre del método con menor RMSE de entre los que hay en el diccionario `metodos` (que ya está preparado).

**Debe salir:** ✔ Ejercicio 5 correcto y una tabla con el RMSE y la precision@3 de los 5 métodos (factorización: 0.860 y 0.180)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A14).

1. ¿Qué método tiene menor RMSE y cuál mayor precision@3?
2. ¿Qué método es el peor en RMSE?
3. ¿Cuántos métodos baten a popularidad en RMSE?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A14, **después de intentarlo**.

In [ ]:
# pista: ya tienes la función factorizar y las funciones de evaluación de los apartados anteriores
P_mf = ...
rmse_mf = ...
p_mf = ...

metodos = {"Popularidad": P_pop, "Contenido": P_cont, "Item-item": P_item, "User-user": P_user, "Factorización": P_mf}
tabla = pd.DataFrame({
    "RMSE": {n: evaluar_rmse(P) for n, P in metodos.items()},
    "precision@3": {n: precision_k(P) for n, P in metodos.items()},
}).round(3)
mejor_rmse = tabla["RMSE"].idxmin()
assert P_mf.shape == (300, 25), "P_mf debe ser 300 x 25"
assert rmse_mf < rmse_pop, "La factorización debería mejorar a popularidad; revisa k=3"
assert abs(p_mf - precision_k(P_mf)) < 1e-9, "p_mf debe ser precision_k(P_mf)"
display(tabla)
print("✔ Ejercicio 5 correcto")

### 👀 A15 · EJEMPLO — La comparación en gráficos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A15).

1. ¿Qué método gana en RMSE y qué método en precision@3?
2. ¿Coincide la clasificación de RMSE y la de precision@3?
3. ¿Por qué no hay que fiarse de diferencias de 0,01 en precision@3?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
tabla["RMSE"].plot.barh(ax=ax[0], color="tab:blue"); ax[0].set_title("Error de la nota (RMSE) — menos es mejor"); ax[0].set_xlabel("estrellas")
tabla["precision@3"].plot.barh(ax=ax[1], color="tab:green"); ax[1].set_title("precision@3 — más es mejor"); ax[1].set_xlabel("proporción de aciertos")
plt.tight_layout(); plt.show()

### ✏️ A16 · MINI-RETO — La función recomendar

**Tu turno:** Escribe `recomendar(id_usuario, n=3, P=P_mf)` que devuelva un `DataFrame` con las `n` experiencias **que el usuario aún no ha valorado** con mayor puntuación prevista, con las columnas `nombre`, `categoria`, `isla` y `nota_prevista`. Prueba con los usuarios 1 y 2 y comprueba que las recomendaciones son distintas. Los `id_usuario` van de 1 al número de usuarios (`len(M)`): ante cualquier otro valor (por ejemplo 0, que en Python apuntaría al último usuario) la función debe lanzar un `ValueError`. **Para el reto del sprint:** el segundo método que se compara con la popularidad es **item-item**, así que allí llamad a `recomendar(..., P=P_item)` (el valor por defecto `P_mf` es solo el de este cuaderno).

**Debe salir:** ✔ Mini-reto correcto y dos tablas con las 3 recomendaciones de los usuarios 1 y 2 (p. ej. Parque Timanfaya, Ruta por Anaga y Zoo y jardines para el usuario 1)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A16).

1. ¿Qué tres experiencias recibe el usuario 1 y cuál es la mejor nota prevista?
2. ¿Por qué `recomendar(0)` lanza un error si en Python `P[-1]` sí existe?
3. ¿Por qué las recomendaciones de los usuarios 1 y 2 son distintas con el mismo modelo?

In [ ]:
# pista: copia P[id_usuario-1], pon -inf donde M[id_usuario-1] sea True, np.argsort(-...) y busca en experiencias por id
def recomendar(id_usuario, n=3, P=P_mf):
    ...

r1, r2 = recomendar(1), recomendar(2)
assert len(r1) == 3 and set(["nombre", "categoria", "isla", "nota_prevista"]) <= set(r1.columns), "Faltan columnas en el resultado"
vistas_1 = set(np.where(M[0])[0] + 1)
assert not (set(r1.index) & vistas_1), "No recomiendes experiencias que el usuario ya valoró"
assert set(r1.index) != set(r2.index), "Los usuarios 1 y 2 deberían recibir recomendaciones distintas"
try:
    recomendar(0)
    raise AssertionError("recomendar(0) debería lanzar ValueError: los usuarios empiezan en 1")
except ValueError:
    pass
print("Usuario 1:"); display(r1)
print("Usuario 2:"); display(r2)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn` y `cosine_similarity` de `scikit-learn` (no se usa ninguna librería especializada en recomendación).
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Fija la semilla `SEMILLA = 42` y lee `valoraciones_experiencias.csv` (`valoraciones`) y `experiencias.csv` (`experiencias`, el catálogo de 25 experiencias).
- **Qué debe salir:**
    - `(2232, 3) (25, 4)` (2 232 valoraciones con 3 columnas y 25 experiencias con 4) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. La matriz usuario × experiencia y la prueba

#### A2 · Ejemplo — Las primeras valoraciones

**Contexto TurisData.** Además del alojamiento, TurisData vende **25 experiencias** (excursiones, talleres, deportes). Hay 2 232 valoraciones de 300 usuarios (de 1 a 5 estrellas). Cada usuario solo ha valorado entre 4 y 11 experiencias: la mayoría de las casillas de la tabla están **vacías**. Queremos decirle a cada cliente *«esto te puede gustar»*.

- **Qué hace:**
    - `valoraciones.head()` muestra las 5 primeras valoraciones: una fila por cada vez que un usuario puntúa una experiencia.
- **Qué debe salir:**
    - Una tabla de 5 filas con `id_usuario`, `id_experiencia` y `valoracion`. El usuario 1 valoró la experiencia 10 con un 1, la 22 con un 4, la 7 con un 2, la 15 con un 5 y la 17 con un 3.
- **Cómo leerlo:**
    - Cada fila es una valoración. No hay una fila por cada pareja usuario-experiencia: solo existen las que el usuario hizo.
- **Si falla:**
    - `NameError: name 'valoraciones' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas valoraciones hay y qué columnas tiene la tabla?* → **2 232** valoraciones y **3** columnas: `id_usuario`, `id_experiencia` y `valoracion`.
2. *¿Qué nota dio el usuario 1 a la experiencia 15?* → **5** estrellas.
3. *¿Por qué no hay una nota para cada pareja usuario-experiencia?* → Porque cada usuario solo ha probado **entre 4 y 11** de las 25 experiencias: las demás casillas están vacías (y justamente son las que queremos predecir).

</details>

#### A3 · Ejemplo — Separar la prueba y construir la matriz

**Idea.** Para saber si un recomendador es bueno hay que **esconderle** algunas valoraciones y ver si las adivina. Reservamos un 20 % al azar como **prueba** (`prueba`) y construimos la matriz solo con el 80 % restante (`entrenamiento`). Las casillas vacías son `NaN`.

- **Qué hace:**
    - `valoraciones.sample(frac=0.2, ...)` elige el 20 % de las valoraciones al azar (`prueba`) y `drop` deja el resto en `entrenamiento`.
    - `entrenamiento.pivot(...)` construye la matriz `R`: una fila por usuario (1 a 300), una columna por experiencia (1 a 25) y la nota en cada casilla; `reindex` asegura que estén todos los usuarios y experiencias.
    - `M = R.notna().values` es una matriz de verdadero/falso: `True` donde el usuario ha valorado esa experiencia.
    - `media_global` es la nota media del entrenamiento. `fila_p`, `col_p` y `real_p` guardan la posición y la nota real de cada valoración de prueba.
    - `evaluar_rmse(P)` es una función: dada una matriz de predicciones `P` (300 × 25), calcula el error sobre las valoraciones de prueba (con las notas limitadas entre 1 y 5).
- **Qué debe salir:**
    - `Matriz: (300, 25) · casillas con dato: 24%` y `Nota media global: 2.97`.
- **Cómo leerlo:**
    - Solo el 24 % de las 7 500 casillas tiene dato en el entrenamiento: la matriz está casi vacía. Es lo normal en recomendación.
- **Si falla:**
    - `NameError: name 'valoraciones' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tamaño tiene la matriz `R` y qué porcentaje de casillas tiene dato?* → **300 × 25** y solo el **24 %** tiene dato.
2. *¿Qué nota media global sale?* → **2,97** estrellas.
3. *¿Por qué se esconde un 20 % de las valoraciones antes de construir la matriz?* → Para **evaluar** de forma honesta: el recomendador intenta adivinar esas notas ocultas y comparamos con las reales. Si las viera al entrenar, no sabríamos si generaliza.

</details>

#### A4 · Ejemplo — El punto de partida: predecir siempre la media

**RMSE** = raíz del error cuadrático medio: *«de media, ¿en cuántas estrellas nos equivocamos?»*. Cuanto más bajo, mejor. Punto de partida: predecir siempre la nota media global.

- **Qué hace:**
    - `np.full((300, 25), media_global)` crea una matriz de predicciones donde todas las casillas valen 2,97 (`P_global`).
    - `evaluar_rmse(P_global)` mide cuánto se equivoca esa predicción sobre la prueba.
- **Qué debe salir:**
    - `RMSE prediciendo siempre la media global: 1.275`.
- **Cómo leerlo:**
    - Un error de 1,275 estrellas, en una escala de 1 a 5, es la cifra que hay que bajar: cualquier recomendador razonable debe mejorarla.
- **Si falla:**
    - `NameError: name 'evaluar_rmse' is not defined` → falta ejecutar el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué RMSE tiene predecir siempre la media global?* → **1,275** estrellas.
2. *¿Qué significa ese número en palabras?* → Que, de media, la predicción se equivoca en **más de una estrella** (en una escala de 1 a 5).
3. *¿Para qué sirve conocer este valor?* → Es la **referencia más básica**: un modelo que no baje de 1,275 no aporta nada frente a decir siempre «un 3».

</details>

## 2. Recomendador por popularidad

#### A5 · Ejercicio 1 — Popularidad: la media de cada experiencia

**Idea.** Recomendar a todos lo mejor valorado. Es simple, no personaliza y **casi siempre es la referencia que hay que batir**. También es la única opción para un usuario nuevo (*arranque en frío*).

- **Qué hace:**
    - `R.mean()` promedia cada **columna** de `R` ignorando los vacíos: la nota media de cada experiencia **solo con el entrenamiento** (`media_item`, 25 valores).
    - `np.tile(media_item.values, (300, 1))` repite ese vector en 300 filas: la predicción para cualquier usuario es la media de la experiencia (`P_pop`, 300 × 25).
    - `evaluar_rmse(P_pop)` mide su error en la prueba (`rmse_pop`).
- **Qué debe salir:**
    - `RMSE popularidad = 1.029` y `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `media_item debe tener 25 valores y P_pop tamaño 300 x 25` → no has hecho la media por experiencia (columnas) o no has repetido el vector en 300 filas.
    - `El RMSE de popularidad debería rondar 1,0; revisa que uses R (solo entrenamiento)` → has calculado la media con `valoraciones` (incluye la prueba) y no con `R`.
    - `NameError: name 'R' is not defined` → falta ejecutar el ejemplo de la matriz.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué RMSE tiene el recomendador por popularidad?* → **1,029**, mejor que predecir siempre la media global (1,275).
2. *¿Por qué mejora al de la media global?* → Porque **no todas las experiencias gustan igual**: usar la media de cada una refleja que unas se valoran mucho más que otras.
3. *¿Por qué hay que calcular `media_item` con `R` y no con todas las valoraciones?* → Porque `R` solo contiene el **entrenamiento**: si la media incluyera la prueba, el modelo «vería» las notas que debe adivinar y el error saldría artificialmente bajo.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
media_item = R.mean()
P_pop = np.tile(media_item.values, (300, 1))
rmse_pop = evaluar_rmse(P_pop)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Las cinco experiencias mejor valoradas

- **Qué hace:**
    - `media_item.sort_values(ascending=False).head(5)` ordena las experiencias de mayor a menor nota media y se queda con las 5 primeras (`top`).
    - `experiencias.set_index("id_experiencia").loc[top.index, "nombre"]` busca sus nombres en el catálogo y `.assign(nota=...)` añade su nota media.
- **Qué debe salir:**
    - Una tabla con 5 experiencias: Parque Timanfaya (4,43), Visita a Garachico (4,21), Observación de cetáceos (3,93), Parapente (3,82) y Astronomía en el Roque (3,77).
- **Cómo leerlo:**
    - Esta es la lista que recibiría **cualquier usuario** con el recomendador por popularidad: la misma para todos.
- **Si falla:**
    - `NameError: name 'media_item' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál es la experiencia mejor valorada y qué nota media tiene?* → **Parque Timanfaya**, con **4,43**.
2. *¿Qué diferencia hay entre la primera y la quinta de la lista?* → 4,43 − 3,77 = **0,66 estrellas**: las cinco primeras están bastante juntas.
3. *¿Qué le recomendaría este método a un usuario al que le encanta el mar?* → **Lo mismo que a todos**: no personaliza. De las cinco, solo «Observación de cetáceos» tiene relación con el mar.

</details>

## 3. Métrica de ranking: precision@k

#### A7 · Ejercicio 2 — La función precision@k

**Idea.** El usuario no ve notas, ve **una lista corta**. Medimos si acertamos con ella: **precision@k** = de las `k` experiencias recomendadas, ¿qué fracción le gustó de verdad (≥ 4 estrellas en la prueba)? Se recomienda solo entre las que el usuario **no ha valorado** en entrenamiento. Se promedia sobre los usuarios que tienen alguna experiencia «que le gusta» en la prueba.

Limitación: si recomendamos algo que le habría gustado pero nunca lo valoró, cuenta como fallo. Por eso los valores son bajos y hay que **compararlos entre métodos**, no leerlos en absoluto.

- **Qué hace:**
    - `relevantes` son las valoraciones de prueba de 4 o 5 estrellas (lo que «le gusta» a cada usuario); el primer `print` cuenta los usuarios que tienen alguna.
    - Dentro de `precision_k`, para cada usuario (`groupby("id_usuario")`): `P[u - 1]` es su fila de predicciones (copiada con `.copy()` para no modificar `P`); `puntuacion[M[u - 1]] = -np.inf` oculta lo que ya valoró.
    - `np.argsort(-puntuacion)[:k] + 1` da las `k` experiencias de mayor puntuación (el `+ 1` pasa de posición a `id_experiencia`).
    - `len(set(recomendadas) & set(g["id_experiencia"])) / k` es la fracción de las `k` que están entre las que le gustaron; se guarda en `aciertos` y la función devuelve la media.
    - La comprobación calcula `p_pop` (popularidad) y `azar` (predicciones al azar) y verifica que sean razonables.
- **Qué debe salir:**
    - `Usuarios evaluables: 115`, `precision@3 popularidad = 0.125 · azar = 0.081` y `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `precision@3 de popularidad debería rondar 0,12; revisa que ocultes lo ya valorado` → no has puesto `-np.inf` en lo ya valorado (`M[u - 1]`) y recomiendas cosas que el usuario ya conoce, o no has sumado 1 al pasar de posición a `id_experiencia`.
    - `Recomendar al azar no debería superar claramente a popularidad; revisa la función` → mismo tipo de fallo: la función no es la correcta.
    - `NameError: name 'M' is not defined` → falta ejecutar el ejemplo de la matriz.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos usuarios son evaluables y por qué no son 300?* → **115**: solo cuentan los usuarios que, en la prueba, valoraron con 4 o 5 estrellas alguna experiencia.
2. *¿Qué precision@3 tiene la popularidad y cuál el azar?* → **0,125** la popularidad y **0,081** el azar.
3. *¿Por qué se pone `-inf` en las experiencias que el usuario ya valoró?* → Para que **no se recomienden cosas que ya conoce**: con `-inf` quedan siempre al final del orden.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
relevantes = prueba[prueba["valoracion"] >= 4]      # lo que "le gusta" a cada usuario
print("Usuarios evaluables:", relevantes["id_usuario"].nunique())

def precision_k(P, k=3):
    aciertos = []
    for u, g in relevantes.groupby("id_usuario"):
        puntuacion = P[u - 1].astype(float).copy()
        puntuacion[M[u - 1]] = -np.inf
        recomendadas = np.argsort(-puntuacion)[:k] + 1
        aciertos.append(len(set(recomendadas) & set(g["id_experiencia"])) / k)
    return float(np.mean(aciertos))
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Recomendador basado en contenido

#### A8 · Ejemplo — Características de las experiencias y residuos

**Idea.** Si a un usuario le gustan las experiencias de *mar*, recomendarle más de *mar*. Cada experiencia se describe por sus **características** (aquí, la categoría) y el usuario por **cuánto le gustan** esas características, calculado con sus propias valoraciones. Ventaja: funciona con experiencias **nuevas** que nadie ha valorado. Límite: solo ve lo que las características describen.

Trabajamos con «residuos»: lo que el usuario valora **por encima o por debajo** de la nota media de esa experiencia (así separamos lo que es *bueno para todos* de lo que es *bueno para él*).

- **Qué hace:**
    - `pd.get_dummies(...)` convierte la categoría de cada experiencia en columnas de 0/1 (una por categoría): `F` es una matriz 25 × 7 (experiencia × categoría).
    - `media_item_v` es la nota media de cada experiencia (las que no tienen dato usan la media global).
    - `C = np.where(M, R.values - media_item_v[None, :], 0.0)` calcula el **residuo** de cada valoración (nota − media de la experiencia) y deja 0 donde no hay valoración.
- **Qué debe salir:**
    - `F (experiencia x categoría): (25, 7) · categorías: 7`.
- **Cómo leerlo:**
    - Las 7 categorías son aventura, bienestar, cultura, familia, gastronomía, mar y naturaleza; naturaleza y mar tienen 6 experiencias cada una y bienestar solo 1.
- **Si falla:**
    - `NameError: name 'media_item' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tamaño tiene `F` y qué significan sus filas y columnas?* → **25 × 7**: una fila por experiencia y una columna por categoría (1 si pertenece, 0 si no).
2. *Si un usuario da un 5 a una experiencia con media 4,0, ¿qué residuo tiene?* → **+1,0** (5 − 4,0): la valora una estrella por encima de lo habitual.
3. *¿Por qué se trabaja con residuos y no con las notas directamente?* → Para separar lo que es **bueno para todos** (una experiencia popular) de lo que es **bueno para ese usuario**: lo que le interesa al recomendar es su desviación respecto a la media.

</details>

#### A9 · Ejercicio 3 — El perfil de gustos de cada usuario

- **Qué hace:**
    - `C @ F` es un producto de matrices: para cada usuario y categoría, **suma sus residuos** en esa categoría (numerador).
    - `M.astype(float) @ F` cuenta, para cada usuario y categoría, **cuántas valoraciones** hizo (denominador); el `+ 3` es la prudencia: con pocas valoraciones, el perfil se queda cerca de 0.
    - El cociente es `perfil_usuario` (300 × 7): positivo = le gusta esa categoría más que a la media; negativo = menos.
- **Qué debe salir:**
    - `✔ Ejercicio 3 correcto`. Por ejemplo, el usuario 2 sale con `mar` ≈ +0,34 y `gastronomia` ≈ +0,17, y con `naturaleza` ≈ −0,26.
- **Si falla:**
    - `perfil_usuario debe ser 300 usuarios x 7 categorías` → has dividido a lo equivocado o has usado `F.T`.
    - `Los valores deben ser residuos pequeños (entre -3 y 3); revisa la división` → te falta dividir entre (nº de valoraciones + 3), o el `+ 3` no está en el denominador.
    - `Sin valoraciones el perfil debe ser 0` → el `+ 3` no está en el denominador (sin él, dividirías 0 entre 0).
    - `NameError: name 'C' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué significa un valor de +0,34 en la columna `mar`?* → Que ese usuario valora las experiencias de **mar** unas 0,34 estrellas **por encima** de su media.
2. *¿Por qué se suma 3 al denominador?* → Es una **prudencia**: con pocas valoraciones en una categoría, el perfil se queda cerca de 0 en vez de fiarse de uno o dos datos.
3. *¿Qué perfil tiene un usuario sin valoraciones en entrenamiento?* → **Todo 0**: no hay residuos que sumar (y el `+ 3` evita dividir entre 0).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
perfil_usuario = (C @ F) / (M.astype(float) @ F + 3)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A10 · Ejemplo — Predicción por contenido y su error

- **Qué hace:**
    - `media_item_v[None, :] + perfil_usuario @ F.T` predice la nota: media de la experiencia + afinidad del usuario con la categoría de esa experiencia (`P_cont`, 300 × 25).
    - El `print` da su RMSE y su precision@3.
- **Qué debe salir:**
    - `RMSE contenido: 1.066 · precision@3: 0.13`.
- **Cómo leerlo:**
    - Compara con popularidad: RMSE 1,029 y precision@3 0,125. El recomendador por contenido **no mejora** en RMSE (1,066 es peor) y casi iguala en precision@3.
- **Si falla:**
    - `NameError: name 'perfil_usuario' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué RMSE y precision@3 da el contenido?* → RMSE **1,066** y precision@3 **0,13**.
2. *¿Es mejor o peor que popularidad (1,029 y 0,125)?* → **Peor** en RMSE (1,066 > 1,029) y **casi igual** en precision@3 (0,13 frente a 0,125).
3. *¿Qué debe cambiar para que el contenido mejore?* → Describir las experiencias con **más características** que la categoría (isla, precio, duración, dificultad…), porque 7 categorías para 25 experiencias son una descripción muy gruesa.

</details>

## 5. Filtrado colaborativo

#### A11 · Ejercicio 4 — Experiencias parecidas al Teide

**Idea.** No hace falta describir las experiencias. Miramos **quién valoró qué**:
- **Entre experiencias (item-item):** dos experiencias son *parecidas* si los mismos usuarios las puntúan por encima o por debajo de lo normal. Para predecir la nota de un usuario a una experiencia, usamos las notas que dio a las experiencias **parecidas**.
- **Entre usuarios (user-user):** dos usuarios son parecidos si valoran igual las mismas experiencias.

La «similitud» es el **coseno** entre dos columnas (o filas) de residuos: 1 = van a la par, 0 = nada que ver, −1 = opuestos.

- **Qué hace:**
    - `cosine_similarity(C.T)` calcula la similitud coseno entre todas las parejas de **columnas** de `C` (las experiencias): `S` es 25 × 25 y simétrica.
    - `np.fill_diagonal(S, 0)` pone la diagonal a 0: una experiencia no se ayuda a sí misma.
    - `np.argsort(-S[0])[:3] + 1` da los ids de las 3 experiencias más parecidas a la 1 (la fila 0, el Ascenso al Teide); `.loc[ids, "nombre"].tolist()` los convierte en nombres.
- **Qué debe salir:**
    - `parecidas al Teide: ['Paseo a caballo', 'Spa y termas', 'Parque acuático']` y `✔ Ejercicio 4 correcto`. Sus similitudes son bajas: 0,19, 0,13 y 0,11.
- **Si falla:**
    - `S debe ser 25 x 25 y simétrica` → has calculado la similitud entre usuarios (`cosine_similarity(C)`, 300 × 300) o entre otras columnas: aquí son las experiencias (`C.T`).
    - `La diagonal de S debe ser 0` → falta `np.fill_diagonal(S, 0)`.
    - `similares_teide: lista con 3 nombres` → has guardado ids (números) y no nombres, o más/menos de 3.
    - `NameError: name 'C' is not defined` → falta el ejemplo de los residuos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuáles son las 3 experiencias más parecidas al Teide y qué similitud tiene la primera?* → **Paseo a caballo, Spa y termas y Parque acuático**; la primera, con similitud **0,19**.
2. *¿Qué indica que las similitudes sean tan bajas (0,19 como máximo)?* → Que las valoraciones de unas experiencias **dicen poco** de las otras: hay mucho ruido y pocos datos por pareja (la matriz está vacía en un 76 %).
3. *¿Por qué se pone la diagonal a 0?* → Porque la similitud de una experiencia consigo misma es 1 y **siempre sería la «más parecida»**: queremos las *otras* más parecidas.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
S = cosine_similarity(C.T)
np.fill_diagonal(S, 0)
ids_similares = np.argsort(-S[0])[:3] + 1
similares_teide = experiencias.set_index("id_experiencia").loc[ids_similares, "nombre"].tolist()
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A12 · Ejemplo — Predicción item-item y user-user

- **Qué hace:**
    - `P_item` predice con las experiencias parecidas: nota media + media **ponderada** (por la similitud `S`) de los residuos que el usuario dio a las experiencias parecidas.
    - `Su = cosine_similarity(C)` es la similitud entre **usuarios** (filas de `C`); `P_user` predice con la nota que dieron los usuarios parecidos.
    - El bucle imprime el RMSE y la precision@3 de cada método.
- **Qué debe salir:**
    - `item-item: RMSE = 0.895 · precision@3 = 0.171` y `user-user: RMSE = 0.865 · precision@3 = 0.148`.
- **Cómo leerlo:**
    - Ambos métodos baten con claridad a popularidad (RMSE 1,029): se **personaliza** de verdad. Item-item acierta mejor la lista (0,171) y user-user la nota (0,865).
- **Si falla:**
    - `NameError: name 'S' is not defined` → falta el ejercicio 4.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuál de los dos tiene menor RMSE y cuál mayor precision@3?* → **User-user** el menor RMSE (0,865) e **item-item** la mayor precision@3 (0,171).
2. *¿Mejoran a popularidad (RMSE 1,029)?* → **Sí**, claramente: 0,895 y 0,865 frente a 1,029.
3. *¿Qué compara la similitud de item-item y qué la de user-user?* → Item-item compara **experiencias** (columnas de `C`, según quién las valoró) y user-user compara **usuarios** (filas, según qué valoraron).

</details>

## 6. Factorización de matrices: gustos ocultos

#### A13 · Ejemplo — La función factorizar y el efecto de k

**Idea.** Supón que cada usuario tiene unos pocos **gustos ocultos** (p. ej. «aventura», «tranquilidad», «cultura») y cada experiencia tiene una **intensidad** en cada uno. La nota esperada se obtiene multiplicando ambos: *nota ≈ media + sesgo del usuario + sesgo de la experiencia + gustos·intensidades*. El algoritmo (aquí, mínimos cuadrados alternos) aprende los gustos y las intensidades a partir de las casillas conocidas y **rellena las vacías**. Es la base de los grandes recomendadores.

- **Qué hace:**
    - `factorizar(R, k, lam, iteraciones)` aprende, para cada usuario, `k` gustos ocultos (`P`) y, para cada experiencia, `k` intensidades (`Q`), junto con un sesgo por usuario (`b_u`) y por experiencia (`b_i`). Repite 30 veces: ajusta los sesgos y luego resuelve por mínimos cuadrados `P` y `Q`. Devuelve la matriz completa de predicciones.
    - `lam` (la regularización) evita que los gustos se ajusten demasiado a pocas notas.
    - El bucle final prueba `k` = 2, 3, 4 y 6 gustos ocultos y escribe el RMSE de cada uno. Tarda unos segundos.
- **Qué debe salir:**
    - `k = 2 gustos ocultos → RMSE = 0.875` · `k = 3 … = 0.860` · `k = 4 … = 0.867` · `k = 6 … = 0.870`.
- **Cómo leerlo:**
    - No hace falta entender cada línea de `factorizar`. Lo importante: el RMSE **mejora hasta k = 3** (0,860) y vuelve a subir con más gustos. Ojo: aquí se compara `k` mirando el error en la prueba; en un caso real habría que elegirlo con una validación aparte.
- **Si falla:**
    - `NameError: name 'evaluar_rmse' is not defined` → falta ejecutar el ejemplo de la matriz.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Con qué k sale el menor RMSE y cuánto vale?* → Con **k = 3**: **0,860**.
2. *¿Qué pasa con el RMSE cuando k pasa de 3 a 6?* → **Empeora un poco** (0,860 → 0,870): con más gustos ocultos el modelo se ajusta más al ruido de unas pocas notas.
3. *¿Qué son los «gustos ocultos» y por qué se llaman así?* → Son `k` números por usuario (y por experiencia) que el algoritmo **descubre solo**; nadie los etiqueta: se llaman «ocultos» porque no son columnas de los datos.

</details>

#### A14 · Ejercicio 5 — La factorización y la tabla comparativa

- **Qué hace:**
    - `factorizar(R, k=3)` devuelve la matriz de predicciones (`P_mf`, 300 × 25); `evaluar_rmse(P_mf)` y `precision_k(P_mf)` miden su error y su precision@3 (`rmse_mf`, `p_mf`).
    - La comprobación reúne los 5 métodos en el diccionario `metodos` y construye `tabla` con el RMSE y la precision@3 de cada uno; `mejor_rmse` es el nombre del método con menor RMSE (`idxmin()`).
- **Qué debe salir:**
    - `✔ Ejercicio 5 correcto` y la tabla (RMSE · precision@3): Popularidad 1,029 · 0,125 / Contenido 1,066 · 0,130 / Item-item 0,895 · 0,171 / User-user 0,865 · 0,148 / Factorización 0,860 · 0,180. `mejor_rmse` vale `'Factorización'`.
- **Si falla:**
    - `P_mf debe ser 300 x 25` → no has guardado en `P_mf` el resultado de `factorizar(R, k=3)`.
    - `La factorización debería mejorar a popularidad; revisa k=3` → has usado otro `k` o has factorizado otra matriz que `R`.
    - `p_mf debe ser precision_k(P_mf)` → `p_mf` no es la precision@3 de `P_mf` (¿has puesto el RMSE?).
    - `NameError: name 'P_cont' is not defined` → faltan ejemplos de arriba (los 5 métodos de `metodos` deben existir).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué método tiene menor RMSE y cuál mayor precision@3?* → **Factorización** en ambos: RMSE **0,860** y precision@3 **0,180**.
2. *¿Qué método es el peor en RMSE?* → **Contenido** (1,066), peor incluso que popularidad (1,029).
3. *¿Cuántos métodos baten a popularidad en RMSE?* → **Tres** (item-item, user-user y factorización); el contenido no.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
P_mf = factorizar(R, k=3)
rmse_mf = evaluar_rmse(P_mf)
p_mf = precision_k(P_mf)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A15 · Ejemplo — La comparación en gráficos

- **Qué hace:**
    - Dibuja dos gráficos de barras horizontales a partir de `tabla`: a la izquierda el RMSE de cada método (menos es mejor) y a la derecha la precision@3 (más es mejor).
- **Qué debe salir:**
    - Dos gráficos con 5 barras cada uno. Izquierda: las barras más cortas son Factorización (0,86) y User-user (0,87); la más larga, Contenido (1,07). Derecha: la más larga es Factorización (0,18); la más corta, Popularidad (0,125).
- **Cómo leerlo:**
    - **Lectura.** Fíjate en que cambiar de métrica cambia la clasificación: predecir bien la nota (RMSE) no es lo mismo que **ordenar bien** la lista (precision@k). Con solo ~115 usuarios evaluables, las diferencias pequeñas en precision@3 son **ruido**: no vendas una mejora de 0,01 al cliente.
- **Si falla:**
    - `NameError: name 'tabla' is not defined` → falta el ejercicio 5.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué método gana en RMSE y qué método en precision@3?* → **Factorización** en las dos.
2. *¿Coincide la clasificación de RMSE y la de precision@3?* → **No del todo**: en RMSE el segundo es user-user (0,865), pero en precision@3 el segundo es item-item (0,171).
3. *¿Por qué no hay que fiarse de diferencias de 0,01 en precision@3?* → Porque solo hay ~**115 usuarios evaluables**: con tan pocos, una diferencia así es **ruido**.

</details>

## 7. Mini-reto: el recomendador de TurisData

#### A16 · Mini-reto — La función recomendar

- **Qué hace:**
    - `if not 1 <= id_usuario <= len(M): raise ValueError(...)` rechaza ids fuera de 1 a 300 (el 0 apuntaría al último usuario por cómo cuenta Python).
    - `puntuacion = P[id_usuario - 1]` es la fila del usuario; `puntuacion[M[id_usuario - 1]] = -np.inf` oculta lo que ya valoró; `np.argsort(-puntuacion)[:n] + 1` da los ids de las `n` mejores.
    - `experiencias.set_index("id_experiencia").loc[ids, ["nombre", "categoria", "isla"]]` busca sus datos y `res["nota_prevista"]` añade la nota prevista (limitada entre 1 y 5, con 2 decimales).
    - La comprobación llama a la función con los usuarios 1 y 2, verifica columnas, que no se repitan experiencias ya valoradas, que las listas sean distintas y que `recomendar(0)` lance `ValueError`.
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y dos tablas. Usuario 1: Parque Timanfaya (4,69), Ruta por Anaga (4,34), Zoo y jardines (3,50). Usuario 2: Observación de cetáceos (4,39), Astronomía en el Roque (4,19), Visita a Garachico (3,96).
- **Cómo leerlo:**
    - Cada usuario recibe una lista distinta (personalizada) y ninguna experiencia que ya haya valorado. Para el reto del sprint, el método que se compara con popularidad es item-item: allí se llama `recomendar(..., P=P_item)`.
- **Si falla:**
    - `Faltan columnas en el resultado` → el DataFrame debe tener `nombre`, `categoria`, `isla` y `nota_prevista`.
    - `No recomiendes experiencias que el usuario ya valoró` → no has puesto `-inf` donde `M[id_usuario - 1]` es `True`.
    - `Los usuarios 1 y 2 deberían recibir recomendaciones distintas` → has usado la misma fila de `P` para los dos (¿te falta el `id_usuario - 1`?).
    - `recomendar(0) debería lanzar ValueError: los usuarios empiezan en 1` → falta el `raise ValueError` al principio de la función.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tres experiencias recibe el usuario 1 y cuál es la mejor nota prevista?* → **Parque Timanfaya, Ruta por Anaga y Zoo y jardines**; la mejor nota prevista es **4,69**.
2. *¿Por qué `recomendar(0)` lanza un error si en Python `P[-1]` sí existe?* → Porque `0 - 1 = -1` apuntaría **al último usuario** sin avisar: devolvería recomendaciones de otra persona. Es mejor fallar.
3. *¿Por qué las recomendaciones de los usuarios 1 y 2 son distintas con el mismo modelo?* → Porque la **fila de predicciones es distinta** (gustos distintos) y además se excluyen las experiencias que cada uno ya valoró.

</details>

## Preguntas de reflexión
1. ¿Qué método usarías para un **huésped nuevo** sin valoraciones? ¿Y para una **experiencia nueva**?
2. El recomendador por contenido salió peor que popularidad en RMSE. ¿Por qué crees que las categorías
   no bastan para explicar los gustos? ¿Qué otras características añadirías?
3. Si TurisData solo recomienda lo más popular, ¿qué experiencias quedarán siempre invisibles? ¿Es justo
   para los proveedores locales pequeños? *(riesgo de sesgo de popularidad)*.
4. ¿Qué datos personales usa este recomendador? ¿Qué debería saber el cliente según el RGPD?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. **Huésped nuevo** (sin valoraciones, el *arranque en frío*): **popularidad**, porque no hay historial con el que personalizar (o contenido, si nos dice sus intereses). **Experiencia nueva**: **contenido**, porque nadie la ha valorado todavía y el colaborativo y la factorización no la conocen; el contenido sí puede describirla por su categoría, isla, precio…
2. Porque la categoría es una descripción **muy gruesa**: 7 categorías para 25 experiencias, y dentro de «mar» o «naturaleza» hay experiencias que gustan a públicos distintos. Se podría añadir isla, precio, duración, dificultad, época del año, tamaño del grupo o una descripción de texto.
3. Quedarían invisibles las experiencias **poco valoradas o nuevas**, justamente las de **proveedores locales pequeños**. Es un círculo vicioso: lo popular se recomienda más, recibe más reservas y más valoraciones, y se vuelve más popular. Para corregirlo se puede mezclar popularidad con personalización o reservar huecos de exploración a las experiencias poco vistas.
4. Usa el **identificador del usuario** y su **historial de valoraciones** (qué experiencias ha puntuado y con qué nota): es un perfil de gustos. Según el RGPD, el cliente debe saber qué datos se usan y con qué finalidad (**transparencia**), poder **acceder, rectificar y borrar** esos datos, y poder **oponerse** a que se elabore un perfil con ellos. Conviene usar solo los datos necesarios (**minimización**).

</details>

## Qué has aprendido
- Popularidad, contenido, colaborativo y factorización: qué aporta y qué exige cada uno.
- Separar entrenamiento y prueba también en recomendación.
- RMSE mide la nota; **precision@k** mide la lista que ve el cliente.
- El arranque en frío se resuelve con popularidad o contenido.